# functional-module-wrap — ex1: wrap F.relu in an nn.Module to make it composable

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `functional-module-wrap`. Running the final beacon cell reports progress against the `PyTorch: functional module wrap` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: functional module wrap` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`functional-module-wrap`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "functional-module-wrap"
DD_SUBTOPIC = "PyTorch: functional module wrap"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## PyTorch: functional vs Module wrap — quick refresher

Every common neural-net building block exists in TWO forms in PyTorch: a stateless functional version (`torch.nn.functional`) and a stateful Module wrapper (`torch.nn`):

```python
import torch.nn.functional as F
import torch.nn as nn

# Functional — call once, no state, no params.
y = F.relu(x)
y = F.linear(x, weight, bias)
y = F.dropout(x, p=0.5, training=True)

# Module — instantiate, owns params/buffers, integrates with nn.Module.
y = nn.ReLU()(x)
y = nn.Linear(in_features=10, out_features=4)(x)
y = nn.Dropout(p=0.5)(x)
```

**When to use `F.`** Stateless ops (`relu`, `softmax`, `gelu`, `cross_entropy`, `pad`). No params. No training/eval mode switching. You just need the function applied.

**When to use `nn.`** When you need:
- Parameters tracked by `model.parameters()` (e.g. `nn.Linear`).
- Train/eval mode switching (`nn.Dropout`, `nn.BatchNorm2d`).
- Composition with `nn.Sequential`.
- `state_dict` serialization.

**The Module is a thin wrapper over the functional.** `nn.ReLU().forward(x)` is literally `F.relu(x)`. The wrapper just adds `nn.Module` registration so it shows up in printouts and state dicts.

**The 'dropout trap'.** `F.dropout(x, p=0.5)` ALWAYS applies dropout — there's no implicit train/eval mode. You must pass `training=self.training` or call from inside an `nn.Dropout` module which handles the flag for you.

### Exercise 1 — wrap F.relu in an nn.Module to make it composable

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the functional-to-Module wrap pattern: implement a stateless `nn.Module` whose `forward` simply calls `F.relu`, matching `nn.ReLU` semantically.
> Keywords: functional, module, relu, compose
> ```

**KCs targeted:** `functional-module-equivalence`, `module-wrap-stateless-fn`

Implement `MyReLU(nn.Module)` whose `forward(x)` returns `F.relu(x)`. This is exactly what `nn.ReLU` does internally — wrap a stateless functional in an `nn.Module` so it composes with `nn.Sequential` and shows up in `state_dict`/`model.modules()` printouts.

Constraints:
- Subclass `nn.Module`.
- Call `super().__init__()` in your `__init__`.
- `forward(x)` must return `F.relu(x)` — do NOT instantiate `nn.ReLU` and delegate (defeats the point).
- No parameters — the module is stateless.

Output: an `nn.Module` subclass that behaves identically to `nn.ReLU` on every input.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class MyReLU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return F.relu(x)


<details><summary>Solution</summary>

```python
import torch.nn as nn
import torch.nn.functional as F

class MyReLU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return F.relu(x)
```

**Three lines, full Module integration.** `super().__init__()` registers the module with the `nn.Module` machinery (children, parameters, state_dict). `forward` delegates to the functional.

**Why a class for a stateless function.** The class form buys you `nn.Sequential` composition, automatic mode propagation (train/eval), and visibility in `model` printouts. The functional form is a flat function — great for one-off use inside a custom `forward`.

**`nn.ReLU`'s actual implementation** is essentially what you just wrote: `super().__init__()` + `forward = F.relu`. (It also accepts an `inplace=True` flag that calls `F.relu_` instead.) Every other stateless `nn.X` (`nn.GELU`, `nn.Softmax`, `nn.Sigmoid`) is the same pattern.

**Gradient comes for free.** Because `F.relu` is autograd-tracked, wrapping it in a Module doesn't break the backward pass. You didn't have to write any backward logic — that's the value of staying inside the functional layer.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()